# 05b — Drift Monitor: Enable Inference Logging & Create Lakehouse Monitor

**Jackson and Johnson HR Digital**

This notebook wires up model drift monitoring for the predictive hiring endpoint so that
any shift in candidate score distributions or prediction patterns triggers an alert.

### What this notebook does
1. Enables inference table auto-capture on `hr-predictive-hiring-endpoint` — every prediction
   request/response is automatically logged to a Delta table
2. Sends warm-up inferences to create and populate the payload table
3. Creates a flattened SQL view over the raw JSON payload table, exposing the 8 feature
   scores and predicted label as typed columns
4. Creates a Unity Catalog **Lakehouse Monitor** with an `InferenceLog` profile on that view,
   using the historical training data (`candidates` gold table) as the baseline distribution
5. Triggers an initial monitor refresh to generate the drift dashboard

### How drift detection works
- **Feature drift**: are the 8 input score distributions shifting vs. baseline training data?
- **Prediction drift**: is the hire/no-hire ratio changing over time?
- **Data quality**: are scores going out of range or going NULL?
- Drift metrics are computed on a daily and weekly granularity and surfaced in an AI/BI dashboard

### Prerequisites
- Notebook `05_train_ml_model.ipynb` has been run — `hr-predictive-hiring-endpoint` is READY
- The endpoint serves `bx4.hrd_2030.hr_predictive_hiring @champion`

**Next notebook:** `06_evaluate_register_agent.ipynb`

## Setup

Install dependencies and configure catalog, schema, and endpoint name.

In [ ]:
%pip install databricks-sdk -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",               "bx4",                                "UC Catalog")
dbutils.widgets.text("schema",                "hrd_2030",                           "UC Schema")
dbutils.widgets.text("endpoint_name",         "hr-predictive-hiring-endpoint",      "Model Serving Endpoint")
dbutils.widgets.text("payload_table_prefix",  "hr_predictive_hiring_monitor",       "Inference Table Prefix")
dbutils.widgets.text("warehouse_id",          "0d3bda4f46281ab5",                   "SQL Warehouse ID")

In [ ]:
# ---------------------------------------------------------------------------
# Imports and config
# ---------------------------------------------------------------------------
import time
import requests
from databricks.sdk import WorkspaceClient

catalog              = dbutils.widgets.get("catalog")
schema               = dbutils.widgets.get("schema")
endpoint_name        = dbutils.widgets.get("endpoint_name")
payload_table_prefix = dbutils.widgets.get("payload_table_prefix")
warehouse_id         = dbutils.widgets.get("warehouse_id")

w            = WorkspaceClient()
current_user = w.current_user.me().user_name

# Derived names
payload_table   = f"{catalog}.{schema}.{payload_table_prefix}_payload"
inference_view  = f"{catalog}.{schema}.hr_predictive_hiring_inference_view"
baseline_table  = f"{catalog}.{schema}.candidates"
assets_dir      = f"/Workspace/Users/{current_user}/monitoring/hr-predictive-hiring"

print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Endpoint         : {endpoint_name}")
print(f"Payload table    : {payload_table}")
print(f"Inference view   : {inference_view}")
print(f"Baseline table   : {baseline_table}")
print(f"Assets dir       : {assets_dir}")

## Step 1 — Enable Inference Logging on the Endpoint

Every prediction request and response will be automatically logged to a Delta table
(`*_payload`) in the target catalog and schema. Existing endpoint configuration
(model version, scale-to-zero, workload size) is preserved.

In [ ]:
# ---------------------------------------------------------------------------
# Enable inference table auto-capture on the serving endpoint.
# The existing served entities are read first so the model version is preserved.
# ---------------------------------------------------------------------------
from databricks.sdk.service.serving import ServedEntityInput, AutoCaptureConfigInput

endpoint = w.serving_endpoints.get(name=endpoint_name)

# Preserve current served entities so we don't change the model version
served_entities = [
    ServedEntityInput(
        entity_name=se.entity_name,
        entity_version=se.entity_version,
        scale_to_zero_enabled=se.scale_to_zero_enabled,
        workload_size=se.workload_size,
    )
    for se in (endpoint.config.served_entities or [])
]

# Check if auto-capture is already on
existing_capture = endpoint.config.auto_capture_config if endpoint.config else None
if existing_capture and existing_capture.enabled:
    payload_table_prefix = existing_capture.table_name_prefix
    payload_table        = f"{catalog}.{schema}.{payload_table_prefix}_payload"
    print(f"✅ Inference logging already enabled on '{endpoint_name}'")
    print(f"   Payload table : {payload_table}")
else:
    print(f"Enabling inference logging on '{endpoint_name}' — updating endpoint config...")
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=served_entities,
        auto_capture_config=AutoCaptureConfigInput(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix=payload_table_prefix,
            enabled=True,
        ),
    )
    print(f"✅ Inference logging enabled on '{endpoint_name}'")
    print(f"   Payload table : {payload_table}")

## Step 2 — Send Warm-Up Inferences

The payload table is created on first inference — Databricks does not create it until the endpoint
receives a request. Send a small batch of representative candidate profiles so the table
exists and has rows before the monitor is created.

In [ ]:
# ---------------------------------------------------------------------------
# Send 5 representative candidate profiles to populate the payload table.
# These mirror the range of scores across the 10 active pipeline candidates.
# ---------------------------------------------------------------------------
host    = w.config.host.rstrip("/")
token   = w.config.token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
url     = f"{host}/serving-endpoints/{endpoint_name}/invocations"

warmup_candidates = [
    # High scorer
    {"education_score": 85, "experience_score": 90, "leadership_score": 85,
     "certification_score": 95, "skills_match_score": 92,
     "industry_relevance_score": 95, "interview_score": 88, "culture_fit": 88},
    # Mid-range
    {"education_score": 72, "experience_score": 75, "leadership_score": 70,
     "certification_score": 68, "skills_match_score": 74,
     "industry_relevance_score": 71, "interview_score": 73, "culture_fit": 69},
    # Low scorer
    {"education_score": 55, "experience_score": 60, "leadership_score": 58,
     "certification_score": 50, "skills_match_score": 62,
     "industry_relevance_score": 55, "interview_score": 60, "culture_fit": 57},
    # Strong leadership / weak certs
    {"education_score": 78, "experience_score": 82, "leadership_score": 92,
     "certification_score": 45, "skills_match_score": 80,
     "industry_relevance_score": 76, "interview_score": 85, "culture_fit": 82},
    # High education / lower experience
    {"education_score": 95, "experience_score": 55, "leadership_score": 65,
     "certification_score": 88, "skills_match_score": 70,
     "industry_relevance_score": 68, "interview_score": 72, "culture_fit": 75},
]

print(f"Sending {len(warmup_candidates)} warm-up inferences to '{endpoint_name}'...")
for i, candidate in enumerate(warmup_candidates, 1):
    resp = requests.post(
        url,
        headers=headers,
        json={"dataframe_records": [candidate]},
        timeout=60,
    )
    resp.raise_for_status()
    pred = resp.json().get("predictions", ["?"])[0]
    label = "RECOMMEND" if pred == 1 else "NOT RECOMMENDED"
    print(f"  Candidate {i}: prediction={pred} ({label})")

print(f"\n✅ {len(warmup_candidates)} inferences sent — payload table will be created shortly")
print("   Waiting 10s for the table to materialise...")
time.sleep(10)

## Step 3 — Inspect Payload Table

Confirm the payload table was created and review its raw schema. The `request` and `response`
columns contain JSON strings — Step 4 creates a view that parses these into typed columns
suitable for drift monitoring.

In [ ]:
# ---------------------------------------------------------------------------
# Inspect the raw payload table schema and row count
# ---------------------------------------------------------------------------
try:
    df_payload = spark.table(payload_table)
    row_count  = df_payload.count()
    print(f"✅ Payload table found: {payload_table}")
    print(f"   Row count : {row_count}")
    print(f"\n   Schema:")
    for field in df_payload.schema.fields:
        print(f"     {field.name:35s} {str(field.dataType)}")
    print("\n   Sample rows:")
    display(df_payload.limit(3))
except Exception as e:
    print(f"⚠️  Payload table not yet visible: {e}")
    print("   The endpoint may still be processing — wait 30s and re-run this cell")

## Step 4 — Create Flattened Inference View

The raw payload table stores requests and responses as JSON strings. This view parses them
into the 8 typed feature columns plus a `prediction` integer — the structure that
Lakehouse Monitoring expects for an `InferenceLog` profile.

In [ ]:
# ---------------------------------------------------------------------------
# Create a flattened view over the raw JSON payload table.
# get_json_object() extracts fields from the request/response JSON strings.
# TRY_CAST avoids failures if a field is missing or malformed.
# ---------------------------------------------------------------------------
view_sql = f"""
CREATE OR REPLACE VIEW {inference_view} AS
SELECT
  date                                                                                   AS timestamp,
  TRY_CAST(get_json_object(response, '$.predictions[0]') AS INT)                        AS prediction,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].education_score')          AS DOUBLE) AS education_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].experience_score')         AS DOUBLE) AS experience_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].leadership_score')         AS DOUBLE) AS leadership_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].certification_score')      AS DOUBLE) AS certification_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].skills_match_score')       AS DOUBLE) AS skills_match_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].industry_relevance_score') AS DOUBLE) AS industry_relevance_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].interview_score')          AS DOUBLE) AS interview_score,
  TRY_CAST(get_json_object(request,  '$.dataframe_records[0].culture_fit')              AS DOUBLE) AS culture_fit,
  status_code,
  execution_time_ms
FROM {payload_table}
WHERE status_code = 200
"""

spark.sql(view_sql)

# Verify the view returns rows
df_view = spark.table(inference_view)
print(f"✅ Inference view created: {inference_view}")
print(f"   Row count : {df_view.count()}")
print("\n   Sample (parsed columns):")
display(df_view.select(
    "timestamp", "prediction",
    "education_score", "experience_score", "leadership_score",
    "interview_score", "culture_fit"
).limit(5))

## Step 5 — Create Lakehouse Monitor

Create a Unity Catalog monitor with an `InferenceLog` profile on the inference view.

- **Baseline**: `candidates` gold table (C001–C010 training data) — the distribution
  the model was trained on
- **Granularities**: daily and weekly windows
- **Slicing**: by `prediction` label to separate hire vs. no-hire distributions
- Databricks automatically generates drift metrics and an AI/BI dashboard

In [ ]:
# ---------------------------------------------------------------------------
# Create the Lakehouse Monitor with an InferenceLog profile.
# The monitor tracks feature and prediction distribution drift over time,
# comparing live inference data against the training baseline.
# ---------------------------------------------------------------------------
from databricks.sdk.service.catalog import (
    MonitorInferenceLog,
    MonitorInferenceLogProblemType,
)

try:
    monitor = w.quality_monitors.create(
        table_name=inference_view,
        assets_dir=assets_dir,
        output_schema_name=f"{catalog}.{schema}",
        inference_log=MonitorInferenceLog(
            timestamp_col="timestamp",
            granularities=["1 day", "1 week"],
            prediction_col="prediction",
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
            # label_col="hired",  # uncomment once ground-truth labels are fed back
        ),
        baseline_table_name=baseline_table,
        slicing_exprs=["prediction"],   # compare hired vs. not-recommended distributions
        warehouse_id=warehouse_id,
    )
    print(f"✅ Lakehouse Monitor created on {inference_view}")
    print(f"   Assets dir      : {assets_dir}")
    print(f"   Baseline table  : {baseline_table}")
    print(f"   Dashboard URL   : {w.config.host}/sql/dashboards/{monitor.dashboard_id}")

except Exception as e:
    if "already exists" in str(e).lower():
        monitor = w.quality_monitors.get(table_name=inference_view)
        print(f"✅ Monitor already exists on {inference_view}")
        print(f"   Dashboard URL : {w.config.host}/sql/dashboards/{monitor.dashboard_id}")
    else:
        raise

## Step 6 — Trigger Initial Monitor Refresh

Run the monitor immediately so baseline drift metrics are computed and the dashboard
is populated. In production the monitor runs on its own schedule; this first run
validates the setup end-to-end.

In [ ]:
# ---------------------------------------------------------------------------
# Trigger an immediate monitor refresh.
# The refresh computes drift metrics and populates the dashboard.
# Large tables can take several minutes; this cell does not block.
# ---------------------------------------------------------------------------
refresh = w.quality_monitors.run_refresh(table_name=inference_view)

print(f"✅ Monitor refresh triggered")
print(f"   Refresh ID     : {refresh.refresh_id}")
print(f"   State          : {refresh.state}")
print(f"\n   The refresh runs asynchronously. Check the monitor dashboard for results:")
print(f"   {w.config.host}/sql/dashboards/{monitor.dashboard_id}")
print(f"\n   Or navigate to: Catalog > {catalog} > {schema} > {inference_view.split('.')[-1]} > Quality")

## Summary & Next Steps

### What was accomplished

| Step | Result |
|---|---|
| Inference logging | Auto-capture enabled on `hr-predictive-hiring-endpoint` |
| Payload table | `bx4.hrd_2030.hr_predictive_hiring_monitor_payload` |
| Warm-up inferences | 5 representative candidates scored to seed the table |
| Inference view | `bx4.hrd_2030.hr_predictive_hiring_inference_view` — parsed columns |
| Monitor profile | InferenceLog — classification, daily + weekly granularity |
| Baseline | `bx4.hrd_2030.candidates` (C001–C010 training distribution) |
| Initial refresh | Triggered asynchronously |

### What drift monitoring detects

| Drift Type | Example signal |
|---|---|
| Feature drift | `experience_score` distribution shifting vs. training baseline |
| Prediction drift | Hire recommendation rate changing week-over-week |
| Data quality | NULL scores, out-of-range values (e.g. score > 100) |
| Model quality | Accuracy drop — requires feeding `hired` labels back (set `label_col`) |

### To enable model quality tracking
Once actual hire decisions are confirmed, join them back to the inference view
and uncomment `label_col="hired"` in the `MonitorInferenceLog` config.

**Proceed to** `06_evaluate_register_agent.ipynb` to build the HireRight AI Agent.